In [1]:
# %%
import pandas as pd
import numpy as np
import warnings
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os

warnings.filterwarnings("ignore")

In [2]:
# 한글 폰트
import matplotlib.pyplot as plt
import warnings

plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

warnings.filterwarnings("ignore")

In [16]:
# 전처리 설정값
# %%
HIGH_NULL_COLS = [
    "착상 전 유전 검사 사용 여부", "PGD 시술 여부", "PGS 시술 여부",
    "난자 해동 경과일", "배아 해동 경과일",
]
COUNT_COLS = [
    "총 시술 횟수", "클리닉 내 총 시술 횟수",
    "IVF 시술 횟수", "DI 시술 횟수",
    "총 임신 횟수", "IVF 임신 횟수", "DI 임신 횟수",
    "총 출산 횟수", "IVF 출산 횟수", "DI 출산 횟수",
]
AGE_MAP = {
    "만18-34세": 26, "만35-37세": 36, "만38-39세": 38,
    "만40-42세": 41, "만43-44세": 43, "만45-50세": 47, "Unknown": 36
}
DONOR_AGE_MAP = {
    "만20세 이하": 19, "만21-25세": 23, "만26-30세": 28,
    "만31-35세": 33, "만36-40세": 38, "만41-45세": 43,
    "알 수 없음": 0, "Unknown": 0
}
CAUSE_COLS = [
    "불임 원인 - 난관 질환", "불임 원인 - 남성 요인", "불임 원인 - 배란 장애",
    "불임 원인 - 여성 요인", "불임 원인 - 자궁경부 문제", "불임 원인 - 자궁내막증",
    "불임 원인 - 정자 농도", "불임 원인 - 정자 면역학적 요인",
    "불임 원인 - 정자 운동성", "불임 원인 - 정자 형태",
]
LOG_COLS = [
    "총 생성 배아 수", "미세주입된 난자 수", "미세주입에서 생성된 배아 수",
    "저장된 배아 수", "수집된 신선 난자 수", "혼합된 난자 수",
    "파트너 정자와 혼합된 난자 수", "미세주입 배아 이식 수",
]

label_encoders = {}

In [25]:
# 전처리 함수 
from sklearn.preprocessing import LabelEncoder

def convert_count(val):
    if pd.isna(val) or val == "Unknown":
        return 0
    if "이상" in str(val):
        return 6
    try:
        return int(str(val).replace("회", "").strip())
    except:
        return 0


def preprocess(df, is_train=True):
    df = df.copy()

    # ── ID 처리 ──
    ids = df["ID"].copy() if "ID" in df.columns else None
    df = df.drop(columns=["ID"], errors="ignore")

    # ── 결측 많은 컬럼 제거 ──
    df = df.drop(columns=[c for c in HIGH_NULL_COLS if c in df.columns])

    # ── 날짜 컬럼 처리 ──
    date_cols = ["난자 채취 경과일", "난자 혼합 경과일", "배아 이식 경과일"]
    for col in date_cols:
        if col in df.columns:
            df[col + "_결측여부"] = df[col].isnull().astype(int)
            df[col] = df[col].fillna(df[col].median() if is_train else 0)

    # ── DI 구조적 결측 처리 ──
    if "시술 유형" in df.columns:
        di_mask = df["시술 유형"] == "DI"

        embryo_cols = [
            "총 생성 배아 수",
            "미세주입에서 생성된 배아 수",
            "저장된 배아 수",
            "미세주입된 난자 수",
            "수집된 신선 난자 수"
        ]

        for col in embryo_cols:
            if col in df.columns:
                df.loc[di_mask, col] = 0
                median_val = df.loc[~di_mask, col].median()
                df.loc[~di_mask, col] = df.loc[~di_mask, col].fillna(median_val)

        df["is_DI"] = (df["시술 유형"] == "DI").astype(int)

    # ── BLASTOCYST feature ──
    if "배아 이식 경과일" in df.columns:
        df["is_blastocyst"] = (df["배아 이식 경과일"] >= 5).astype(int)
        df["is_early_transfer"] = (df["배아 이식 경과일"] <= 3).astype(int)

        def transfer_stage(x):
            if pd.isna(x):
                return "Unknown"
            elif x <= 3:
                return "early"
            elif x <= 4:
                return "mid"
            else:
                return "blast"

        df["배아_이식_stage"] = df["배아 이식 경과일"].apply(transfer_stage)
        df["stage_early"] = (df["배아_이식_stage"] == "early").astype(int)
        df["stage_mid"]   = (df["배아_이식_stage"] == "mid").astype(int)
        df["stage_blast"] = (df["배아_이식_stage"] == "blast").astype(int)

    # ── 숫자형 결측 처리 ──
    num_cols = [c for c in df.select_dtypes(include="number").columns if c != "임신 성공 여부"]
    df[num_cols] = df[num_cols].fillna(0)

    # ── 범주형 결측 처리 ──
    obj_cols = df.select_dtypes(include="object").columns
    df[obj_cols] = df[obj_cols].fillna("Unknown")

    # ── 횟수형 처리 ──
    for col in COUNT_COLS:
        if col in df.columns:
            df[col] = df[col].apply(convert_count)

    # ── 나이 매핑 ──
    if "시술 당시 나이" in df.columns:
        df["시술 당시 나이"] = df["시술 당시 나이"].map(AGE_MAP).fillna(36)

    for col in ["난자 기증자 나이", "정자 기증자 나이"]:
        if col in df.columns:
            df[col] = df[col].map(DONOR_AGE_MAP).fillna(0)

    # ── IVF / DI 파생 변수 ──
    if "IVF 시술 횟수" in df.columns and "DI 시술 횟수" in df.columns:
        df["IVF_DI_시술_합산"] = df["IVF 시술 횟수"] + df["DI 시술 횟수"]
        df["IVF_시술_비율"] = df["IVF 시술 횟수"] / (df["IVF_DI_시술_합산"] + 1e-6)

    if "IVF 임신 횟수" in df.columns and "DI 임신 횟수" in df.columns:
        df["IVF_DI_임신_합산"] = df["IVF 임신 횟수"] + df["DI 임신 횟수"]
        df["IVF_임신_비율"] = df["IVF 임신 횟수"] / (df["IVF_DI_임신_합산"] + 1e-6)

    if "IVF 출산 횟수" in df.columns and "DI 출산 횟수" in df.columns:
        df["IVF_DI_출산_합산"] = df["IVF 출산 횟수"] + df["DI 출산 횟수"]
        df["IVF_출산_비율"] = df["IVF 출산 횟수"] / (df["IVF_DI_출산_합산"] + 1e-6)

    if "IVF_DI_시술_합산" in df.columns and "IVF_DI_임신_합산" in df.columns:
        df["시술_대비_임신_비율"] = df["IVF_DI_임신_합산"] / (df["IVF_DI_시술_합산"] + 1e-6)
    # 🔥 👉 여기 추가 (같은 들여쓰기 레벨)
    if "총 생성 배아 수" in df.columns and "수집된 신선 난자 수" in df.columns:
        df["배아_생성_효율"] = df["총 생성 배아 수"] / (df["수집된 신선 난자 수"] + 1e-6)
    # 🔥 추가 feature
    if "이식된 배아 수" in df.columns and "총 생성 배아 수" in df.columns:
        df["이식_효율"] = df["이식된 배아 수"] / (df["총 생성 배아 수"] + 1e-6)
        
    # ── 불임 원인 개수 ──
    cause_exist = [c for c in CAUSE_COLS if c in df.columns]
    if cause_exist:
        df["불임_원인_개수"] = df[cause_exist].sum(axis=1)

    # ── 배아 사용 조합 ──
    if all(c in df.columns for c in ["동결 배아 사용 여부", "신선 배아 사용 여부", "기증 배아 사용 여부"]):
        df["배아_사용_조합"] = (
            df["동결 배아 사용 여부"].fillna(0).astype(int).astype(str) +
            df["신선 배아 사용 여부"].fillna(0).astype(int).astype(str) +
            df["기증 배아 사용 여부"].fillna(0).astype(int).astype(str)
        )

    # ── Label Encoding ──
    for col in df.select_dtypes(include="object").columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    return df, ids

In [26]:
# ════════════════════════════════════════════════════════════
# 데이터 로드 + 분리 구조 (모델별 feature 분리 포함)
# ════════════════════════════════════════════════════════════
import pandas as pd
train  = pd.read_csv("data/train.csv")
test   = pd.read_csv("data/test.csv")
sample = pd.read_csv("data/sample_submission.csv")

# 전처리
train_df, _       = preprocess(train, is_train=True)
test_df, test_ids = preprocess(test,  is_train=False)

# X, y 분리
X = train_df.drop("임신 성공 여부", axis=1)
y = train_df["임신 성공 여부"]

X_submit = test_df.drop(columns=["임신 성공 여부"], errors="ignore")

# train/val split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 🔥 여기 핵심 (모델별 데이터 분리)
X_train_lgb = X_train.copy()
X_val_lgb   = X_val.copy()

X_train_full = X_train.copy()
X_val_full   = X_val.copy()

# class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"사용 피처 수: {X_train.shape[1]}")
print(f"scale_pos_weight: {scale_pos_weight:.4f}")


사용 피처 수: 83
scale_pos_weight: 2.8707


In [33]:
# LGB 전용 feature 
drop_cols = [
    "is_blastocyst",
    "is_early_transfer",
    "배아_이식_stage",
    "stage_early",
    "stage_mid",
    "stage_blast",
    "blast_age_interaction",

    # 🔥 추가 (중복 제거)
    "총 생성 배아 수",
    "수집된 신선 난자 수"
]

X_train_lgb = X_train_lgb.drop(columns=drop_cols, errors="ignore")
X_val_lgb   = X_val_lgb.drop(columns=drop_cols, errors="ignore")

In [28]:
best_lgb_params = {
    "n_estimators":      400,
    "learning_rate":     0.04956090868322492,
    "num_leaves":        112,
    "max_depth":         4,
    "min_child_samples": 26,
    "subsample":         0.7523886151586271,
    "colsample_bytree":  0.6353997510251372,
    "reg_alpha":         5.682819774457299,
    "reg_lambda":        8.955144228161172,
    "scale_pos_weight":  scale_pos_weight,
    "random_state":      42,
    "n_jobs":            -1,
    "verbose":           -1,
}

In [29]:
# 🔥 여기부터 CV 코드 시작
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n🔥 Fold {fold+1}")

    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

    # 🔥 drop_cols 적용
    X_tr_lgb = X_tr.drop(columns=drop_cols, errors="ignore")
    X_va_lgb = X_va.drop(columns=drop_cols, errors="ignore")

    model = lgb.LGBMClassifier(**best_lgb_params)
    model.fit(X_tr_lgb, y_tr)

    val_proba = model.predict_proba(X_va_lgb)[:, 1]
    oof_preds[val_idx] = val_proba

    fold_auc = roc_auc_score(y_va, val_proba)
    print(f"Fold {fold+1} AUC: {fold_auc:.4f}")

# 전체 성능
cv_auc = roc_auc_score(y, oof_preds)
print(f"\n🔥 CV AUC: {cv_auc:.4f}")


🔥 Fold 1
Fold 1 AUC: 0.7378

🔥 Fold 2
Fold 2 AUC: 0.7427

🔥 Fold 3
Fold 3 AUC: 0.7398

🔥 Fold 4
Fold 4 AUC: 0.7377

🔥 Fold 5
Fold 5 AUC: 0.7408

🔥 CV AUC: 0.7397


In [31]:
#CV 기반 Threshold Tuning
from sklearn.metrics import f1_score, precision_score, recall_score

print("\n🔥 CV 기반 Threshold Tuning")

for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    preds = (oof_preds >= t).astype(int)
    
    print(
        f"t={t:.2f} | "
        f"F1={f1_score(y, preds):.4f} | "
        f"P={precision_score(y, preds):.4f} | "
        f"R={recall_score(y, preds):.4f}"
    )


🔥 CV 기반 Threshold Tuning
t=0.30 | F1=0.4902 | P=0.3278 | R=0.9715
t=0.40 | F1=0.5063 | P=0.3503 | R=0.9133
t=0.50 | F1=0.5172 | P=0.3852 | R=0.7867
t=0.60 | F1=0.4867 | P=0.4376 | R=0.5480
t=0.70 | F1=0.3316 | P=0.5007 | R=0.2479


In [32]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

import lightgbm as lgb
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n🔥 Fold {fold+1}")

    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

    # LGB
    X_tr_lgb = X_tr.drop(columns=drop_cols, errors="ignore")
    X_va_lgb = X_va.drop(columns=drop_cols, errors="ignore")

    model_lgb = lgb.LGBMClassifier(**best_lgb_params)
    model_lgb.fit(X_tr_lgb, y_tr)
    oof_lgb[val_idx] = model_lgb.predict_proba(X_va_lgb)[:, 1]

    # CAT
    model_cat = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        verbose=0,
        random_state=42
    )
    model_cat.fit(X_tr, y_tr)
    oof_cat[val_idx] = model_cat.predict_proba(X_va)[:, 1]

    # XGB
    model_xgb = XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )
    model_xgb.fit(X_tr, y_tr)
    oof_xgb[val_idx] = model_xgb.predict_proba(X_va)[:, 1]

# 🔥 단일 모델 성능
print("\n--- Single Model AUC ---")
print("LGB:", roc_auc_score(y, oof_lgb))
print("CAT:", roc_auc_score(y, oof_cat))
print("XGB:", roc_auc_score(y, oof_xgb))


🔥 Fold 1

🔥 Fold 2

🔥 Fold 3

🔥 Fold 4

🔥 Fold 5

--- Single Model AUC ---
LGB: 0.7397456742488278
CAT: 0.7396968497515173
XGB: 0.7390438407634244


In [35]:
# 🔥 앙상블 !  가중 평균 (초기값)
w_lgb, w_cat, w_xgb = 0.5, 0.3, 0.2

ensemble_preds = (
    oof_lgb * w_lgb +
    oof_cat * w_cat +
    oof_xgb * w_xgb
)

print("\n🔥 Ensemble AUC:", roc_auc_score(y, ensemble_preds))


🔥 Ensemble AUC: 0.7399993679004517


In [36]:
best_auc = 0
best_weights = None

for wl in np.arange(0.3, 0.7, 0.1):
    for wc in np.arange(0.2, 0.5, 0.1):
        wx = 1 - wl - wc
        if wx < 0:
            continue

        preds = wl * oof_lgb + wc * oof_cat + wx * oof_xgb
        auc = roc_auc_score(y, preds)

        if auc > best_auc:
            best_auc = auc
            best_weights = (wl, wc, wx)

print("\n🔥 Best AUC:", best_auc)
print("🔥 Best weights (LGB, CAT, XGB):", best_weights)


🔥 Best AUC: 0.7400220664487382
🔥 Best weights (LGB, CAT, XGB): (np.float64(0.5), np.float64(0.4000000000000001), np.float64(0.09999999999999992))


In [37]:
# 테스트 예측용
test_preds_lgb = np.zeros(len(X_submit))
test_preds_cat = np.zeros(len(X_submit))
test_preds_xgb = np.zeros(len(X_submit))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n🔥 Fold {fold+1}")

    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]

    # LGB
    X_tr_lgb = X_tr.drop(columns=drop_cols, errors="ignore")
    X_test_lgb = X_submit.drop(columns=drop_cols, errors="ignore")

    model_lgb = lgb.LGBMClassifier(**best_lgb_params)
    model_lgb.fit(X_tr_lgb, y_tr)
    test_preds_lgb += model_lgb.predict_proba(X_test_lgb)[:, 1] / 5

    # CAT
    model_cat = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        verbose=0,
        random_state=42
    )
    model_cat.fit(X_tr, y_tr)
    test_preds_cat += model_cat.predict_proba(X_submit)[:, 1] / 5

    # XGB
    model_xgb = XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )
    model_xgb.fit(X_tr, y_tr)
    test_preds_xgb += model_xgb.predict_proba(X_submit)[:, 1] / 5


🔥 Fold 1

🔥 Fold 2

🔥 Fold 3

🔥 Fold 4

🔥 Fold 5


In [38]:
# 최적 가중치
w_lgb, w_cat, w_xgb = 0.5, 0.4, 0.1

final_preds = (
    test_preds_lgb * w_lgb +
    test_preds_cat * w_cat +
    test_preds_xgb * w_xgb
)

In [45]:
submission = pd.read_csv("data/sample_submission.csv")

submission = submission[["ID"]]

# 🔥 여기 바꿔
submission["probability"] = final_preds

submission.to_csv("submission.csv", index=False)

* 윤재님 파일 다운받아서 그대로 재현했을때 [앙상블 Val AUC]: 0.7374 (LB 점수: 0.7412775224) 확보됨 
이것을 베이스라인으로 할 것임

1. 검증 방법:
train_test_split (test_size=0.2, stratify=y)

2. 사용 피처 수:
X_train.shape[1] (85개)

3. threshold:
0.5 (기본값, 튜닝 없음)

실험 계속. : feature engineering